# Script 16 — KDM-Biological Age outlier sensitivity analysis

The Klemera-Doubal estimator is an unbounded precision-weighted projection, so
extreme biomarker combinations extrapolate without constraint. In the present
adult sample (chronological age 18–80 years) the computed biological age ranges
from −0.4 to 198.8 years, with 163 participants above 100 years, 46 above 120
years, and a maximum advancement of +132.8 years. These values are retained in
the primary analysis; trimming after the estimates were known would be a
post-hoc decision, so trimming is reported as a sensitivity analysis.

The notebook refits the supportive weighted least-squares models after
excluding participants beyond the 0.5th and 99.5th percentiles of
KDM-Biological Age, and compares the coefficients with the primary estimates.

| | |
|---|---|
| KDM-BA 0.5th percentile | 8.65 years |
| KDM-BA 99.5th percentile | 115.86 years |
| Flagged among adults with a computed KDM-BA | 108 / 10,734 (1.0%) |
| Flagged within the depression analysis set | 63 / 9,026 |
| Flagged within the 60+ subset | 28 / 2,929 |

All figures in this table are recomputed in the cells below; the table is a
summary, not the source.

The notebook also writes an augmented Mplus data file carrying an outlier flag,
together with two input files that reproduce the primary structural equation
models on the trimmed sample, so that the full mediation decomposition can be
checked in Mplus.

## Inputs

- `data/processed/8_nhanes_analytic.csv`
- `data/mplus/nhanes_sem.dat` and `nhanes_sem_vars.txt`

## Outputs

- `result/supplementary/SuppTableS11_KDM_Outlier_Sensitivity.csv` and `.xlsx`
- `data/mplus/nhanes_sem_trimmed.dat`
- `scripts/9_mplus_models/S1_depression_trimmed.inp`
- `scripts/9_mplus_models/S1_cognition_trimmed.inp`

## Step 0 — Imports and paths

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE   = find_root()
PROC   = BASE / 'data' / 'processed'
MPLUS  = BASE / 'data' / 'mplus'
MODELS = BASE / 'scripts' / '9_mplus_models'
R_SUPP = BASE / 'result' / 'supplementary'
R_SUPP.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook runs against the code on disk.
import importlib
import wls_utils, nature_palette
import population; importlib.reload(population)
for _m in (nature_palette, wls_utils,):
    importlib.reload(_m)

from wls_utils import wls_fit

from population import ADULT_MIN_AGE, OLD_MIN_AGE, YOUNG, OLD, POOLED
from nature_palette import EXP_ORDER
from nature_palette import fmt_ci as _fmt_ci, age_label
EXPOSURES = list(EXP_ORDER)   # RA, IS, IV
EXP_NAME  = {'IV': 'Intradaily variability', 'IS': 'Interdaily stability',
             'RA': 'Relative amplitude'}


## Step 1 — Load data, derive outcomes, define the outlier flag

In [ ]:
df = pd.read_csv(PROC / '8_nhanes_analytic.csv', low_memory=False)
rename = {
    'RIDAGEYR': 'AGE', 'RIAGENDR': 'SEX', 'BMXBMI': 'BMI',
    'DMDEDUC2': 'EDU', 'Wear_Time': 'WEAR_T', 'SLD010H': 'SLEEP_H',
    'KDM_BA_advance': 'KDM_BA_ADV', 'SII_log': 'SII_LOG',
}
df = df.rename(columns=rename)
assert 'WTMEC4YR' in df.columns, 'run script 8: WTMEC4YR missing'
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'
df = df[df['AGE'] >= ADULT_MIN_AGE].copy()   # the analysis population (population.py)

dpq_cols = [f'DPQ{i:03d}' for i in [10, 20, 30, 40, 50, 60, 70, 80, 90]]
df['PHQ9'] = df[dpq_cols].replace({7: np.nan, 9: np.nan}).sum(axis=1, min_count=9)

elig_60 = df['AGE'] >= OLD_MIN_AGE
for src, dst in [('memory_score', 'MEM_Z'), ('fluency_score', 'FLUEN_Z'),
                 ('speed_score', 'SPEED_Z')]:
    raw = df[src]
    s   = raw.loc[elig_60].dropna()
    df[dst] = np.where(elig_60, (raw - s.mean()) / s.std(), np.nan)
df['COG_Z'] = df[['MEM_Z', 'FLUEN_Z', 'SPEED_Z']].mean(axis=1, skipna=False)

# Outlier flag: below the 0.5th or above the 99.5th percentile of KDM-BA.
P_LO, P_HI = 0.005, 0.995
kdm = df['KDM_BA'].dropna()
lo, hi = kdm.quantile(P_LO), kdm.quantile(P_HI)
df['KDM_OUTLIER'] = ((df['KDM_BA'] < lo) | (df['KDM_BA'] > hi)).fillna(False)

print(f'KDM-BA computed for n = {len(kdm):,}')
print(f'  {P_LO*100:.1f}th percentile  = {lo:8.2f} years')
print(f'  {P_HI*100:.1f}th percentile  = {hi:8.2f} years')
print(f'  observed range     = {kdm.min():.1f} to {kdm.max():.1f} years')
print(f'  flagged as outlier = {int(df["KDM_OUTLIER"].sum()):,} '
      f'({100*df["KDM_OUTLIER"].sum()/len(kdm):.2f}% of those with a value)')

## Step 2 — Helpers (identical machinery to scripts 7 and 8)

In [ ]:
M1_COVS  = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']
MED_VARS = ['KDM_BA_ADV', 'SII_LOG']

def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def fmt_ci(est, lo_, hi_, dp=3):
    return _fmt_ci(est, lo_, hi_, dp)   # minus kept, plus omitted

def weighted_mean_sd(x, w):
    wt = np.sum(w)
    mu = np.sum(x * w) / wt
    return float(mu), float(np.sqrt(np.sum(w * (x - mu) ** 2) / wt))

def run_path(data, exposure, outcome_col, extras=(), age_min=ADULT_MIN_AGE):
    """Survey-weighted WLS with HC3 SE, survey-weighted standardisation of the
    exposure, the outcome, and any mediator extras, on the given sample."""
    need = [exposure, outcome_col, 'WTMEC4YR'] + M1_COVS + list(extras)
    ana  = data[(data['AGE'] >= age_min) & data[need].notna().all(axis=1)]
    if len(ana) < 50:
        return None
    w = ana['WTMEC4YR'].values
    mu_x, sd_x = weighted_mean_sd(ana[exposure].values, w)
    mu_y, sd_y = weighted_mean_sd(ana[outcome_col].values, w)
    cols = [(ana[exposure].values - mu_x) / sd_x]
    for e in extras:
        mu_m, sd_m = weighted_mean_sd(ana[e].values, w)
        cols.append((ana[e].values - mu_m) / sd_m)
    cols += [ana[c].values for c in M1_COVS]
    fit = wls_fit(np.column_stack(cols),
                  (ana[outcome_col].values - mu_y) / sd_y,
                  w=w, se='hc3', add_const=True)
    return dict(n=int(fit['n']), beta=float(fit['params'][1]),
                ci_lo=float(fit['ci_lo'][1]), ci_hi=float(fit['ci_hi'][1]),
                p=float(fit['p'][1]))

## Step 3 — Refit every supportive model with and without the KDM-BA tails

In [ ]:
# Every rhythm metric through every supportive specification, with and
# without the KDM-BA tails.
SPECS = [
    ('{m} -> KDM-BA advancement (a1)',       'KDM_BA_ADV', (),       ADULT_MIN_AGE),
    ('{m} -> log SII (a2)',                  'SII_LOG',    (),       ADULT_MIN_AGE),
    ('{m} -> PHQ-9, total (c)',              'PHQ9',       (),       ADULT_MIN_AGE),
    ("{m} -> PHQ-9, direct (c')",            'PHQ9',       MED_VARS, ADULT_MIN_AGE),
    ('{m} -> cognition, total (c)',          'COG_Z',      (),       OLD_MIN_AGE),
    ("{m} -> cognition, direct (c')",        'COG_Z',      MED_VARS, OLD_MIN_AGE),
]

trimmed = df[~df['KDM_OUTLIER']]
rows = []
for exp in EXPOSURES:
    for label_t, outcome, extras, amin in SPECS:
        label = label_t.format(m=exp)
        full = run_path(df,      exp, outcome, extras, amin)
        trim = run_path(trimmed, exp, outcome, extras, amin)
        if full is None or trim is None:
            continue
        rows.append({
            'Model': label,
            'n (primary)':    f"{full['n']:,}",
            'beta (primary)': fmt_ci(full['beta'], full['ci_lo'], full['ci_hi']),
            'p (primary)':    fmt_p(full['p']),
            'n (trimmed)':    f"{trim['n']:,}",
            'beta (trimmed)': fmt_ci(trim['beta'], trim['ci_lo'], trim['ci_hi']),
            'p (trimmed)':    fmt_p(trim['p']),
            'Delta beta':     f"{trim['beta'] - full['beta']:.3f}",
            'Same direction and inference':
                'yes' if (np.sign(trim['beta']) == np.sign(full['beta'])
                          and (trim['p'] < 0.05) == (full['p'] < 0.05)) else 'NO',
        })

sens = pd.DataFrame(rows)
print(sens.to_string(index=False))
print()
n_same = (sens['Same direction and inference'] == 'yes').sum()
print(f'Models preserving both direction and inference: {n_same} / {len(sens)}')
print(f'Largest absolute shift in beta: '
      f'{sens["Delta beta"].map(lambda s: abs(float(s))).max():.3f}')
disagree = sens[sens['Same direction and inference'] == 'NO']
if len(disagree):
    print('\nDISAGREEMENTS -- these change under trimming and must be reported:')
    print(disagree[['Model', 'beta (primary)', 'p (primary)',
                    'beta (trimmed)', 'p (trimmed)']].to_string(index=False))


## Step 4 — Write the supplementary table

In [ ]:
sens.to_csv(R_SUPP / 'SuppTableS11_KDM_Outlier_Sensitivity.csv', index=False)
try:
    with pd.ExcelWriter(R_SUPP / 'SuppTableS11_KDM_Outlier_Sensitivity.xlsx',
                        engine='openpyxl') as xw:
        sens.to_excel(xw, sheet_name='Supp Table S11', index=False)
        pd.DataFrame({'Notes': [
            'The Klemera-Doubal estimator is an unbounded precision-weighted projection, so extreme '
            'biomarker combinations extrapolate without constraint.',
            f'Observed KDM-Biological Age range in adults aged {ADULT_MIN_AGE}-80 years: {kdm.min():.1f} to {kdm.max():.1f} years.',
            f'Trimming rule: exclude below the 0.5th percentile ({lo:.2f} years) or above the 99.5th '
            f'percentile ({hi:.2f} years) of the KDM-Biological Age distribution.',
            f'Participants flagged: {int(df["KDM_OUTLIER"].sum()):,} of {len(kdm):,} with a computed value.',
            'All participants are retained in the primary analysis. Trimming after the estimates were known '
            'would be a post-hoc decision, so this is reported as a sensitivity analysis rather than as the '
            'primary specification.',
            'Estimation: survey-weighted least squares with HC3 heteroscedasticity-consistent standard errors, '
            'matching the supportive models in Tables 2 and 3.',
            'The full mediation decomposition under trimming is checked in Mplus using the trimmed data file '
            'and input files written by this notebook.',
        ]}).to_excel(xw, sheet_name='Notes', index=False)
    print(f'Wrote {R_SUPP / "SuppTableS11_KDM_Outlier_Sensitivity.xlsx"}')
except Exception as e:
    print(f'(xlsx skipped: {e})')
print(f'Wrote {R_SUPP / "SuppTableS11_KDM_Outlier_Sensitivity.csv"}')
# Move outputs under the unnumbered file names to _superseded/.
for stale in [R_SUPP / 'SuppTable_KDM_Outlier_Sensitivity.csv',
              R_SUPP / 'SuppTable_KDM_Outlier_Sensitivity.xlsx']:
    if stale.exists():
        dest = BASE / '_superseded' / stale.name
        dest.parent.mkdir(exist_ok=True)
        stale.rename(dest)
        print(f'superseded -> _superseded/{stale.name}')


## Step 5 — Write the trimmed Mplus data file and input files

The trimmed data file appends one column, `KDMOUT`, coded 1 for participants
beyond the KDM-Biological Age trimming bounds and 0 otherwise. The two input
files reproduce the primary models with `SUBPOPULATION = ADULT20 EQ 1 AND KDMOUT EQ 0`, so the
mediation decomposition can be re-estimated on the trimmed sample without
altering any other model specification.

In [ ]:
# Column names come from the manifest written by script 9 and are checked
# against the column count of the .dat.
import re as _re
_manifest = (MPLUS / 'nhanes_sem_vars.txt').read_text()
COLS = [m.group(1) for m in
        _re.finditer(r'^#\s+\d+\s+(\w+)\s*$', _manifest, _re.M)]
_ncol = len(open(MPLUS / 'nhanes_sem.dat').readline().split())
assert len(COLS) == _ncol, (
    f'manifest lists {len(COLS)} columns but the .dat has {_ncol}. '
    'Re-run 9_mplus_prep.')
print(f'read {len(COLS)} column names from the manifest')
MISSING = -9999

dat = pd.read_csv(MPLUS / 'nhanes_sem.dat', sep=r'\s+', header=None, names=COLS)
flag = df.set_index('SEQN')['KDM_OUTLIER']
dat['KDMOUT'] = dat['SEQN'].map(flag).fillna(False).astype(int)
print(f'Mplus rows = {len(dat):,}; flagged KDMOUT = 1 for {int(dat["KDMOUT"].sum()):,}')

int_cols = {'SEQN', 'STRATUM', 'PSU', 'SEX', 'KDMOUT'}
out_path = MPLUS / 'nhanes_sem_trimmed.dat'
with open(out_path, 'w') as f:
    for row in dat.itertuples(index=False):
        f.write(' '.join(
            (f'{int(v):d}' if c in int_cols else f'{float(v):.6f}')
            for c, v in zip(dat.columns, row)) + '\n')
print(f'Wrote {out_path}')

# NAMES is built from the same column list as the .dat.
_wrapped = []
for _i in range(0, len(dat.columns), 9):
    _wrapped.append('        ' + ' '.join(dat.columns[_i:_i + 9]))
NAMES = '\n'.join(_wrapped) + ';'

COMMON = f'''
DATA:
    FILE = ../../data/mplus/nhanes_sem_trimmed.dat;

VARIABLE:
    NAMES =
{NAMES}
'''

dep_inp = f'''TITLE:
    Sensitivity S1 - depression dual-mediator SEM, KDM-BA tails trimmed

    Identical to M1_depression.inp except that participants beyond the 0.5th
    ({lo:.2f} years) or 99.5th ({hi:.2f} years) percentile of KDM-Biological Age
    are excluded via SUBPOPULATION (domain analysis -- all strata and PSUs
    stay in the variance estimation, which USEOBSERVATIONS would break).
    All other specification is unchanged.
{COMMON}
    USEVARIABLES =
        IV BA_ADV SII_LOG PHQ9
        AGE SEX BMI EDU WEAR_T SLEEP_H;

    SUBPOPULATION = ADULT20 EQ 1 AND KDMOUT EQ 0;

    MISSING = ALL(-9999);
    IDVARIABLE = SEQN;
    STRATIFICATION = STRATUM;
    CLUSTER        = PSU;
    WEIGHT         = WT4YR;

ANALYSIS:
    TYPE        = COMPLEX;
    ESTIMATOR   = MLR;
    INFORMATION = OBSERVED;

MODEL:
    BA_ADV  ON IV (a1)
               AGE SEX BMI EDU WEAR_T SLEEP_H;
    SII_LOG ON IV (a2)
               AGE SEX BMI EDU WEAR_T SLEEP_H;
    PHQ9 ON IV       (cp)
            BA_ADV   (b1)
            SII_LOG  (b2)
            AGE SEX BMI EDU WEAR_T SLEEP_H;
    BA_ADV WITH SII_LOG;

MODEL INDIRECT:
    PHQ9 IND IV;
    PHQ9 IND BA_ADV IV;
    PHQ9 IND SII_LOG IV;

MODEL CONSTRAINT:
    NEW(ind_BA ind_SII total_ind total);
    ind_BA    = a1 * b1;
    ind_SII   = a2 * b2;
    total_ind = ind_BA + ind_SII;
    total     = cp + total_ind;

OUTPUT:
    STDYX CINTERVAL;
'''

cog_inp = f'''TITLE:
    Sensitivity S1 - cognition dual-mediator SEM, KDM-BA tails trimmed

    Identical to M1_cognition.inp except that participants beyond the 0.5th
    ({lo:.2f} years) or 99.5th ({hi:.2f} years) percentile of KDM-Biological Age
    are excluded. Exclusion and the 60+ restriction are combined in one
    SUBPOPULATION statement (Mplus forbids USEOBSERVATIONS alongside it,
    and domain analysis is the correct subsetting under TYPE = COMPLEX).
{COMMON}
    USEVARIABLES =
        IV BA_ADV SII_LOG
        MEM_Z FLUEN_Z SPEED_Z
        AGE SEX BMI EDU WEAR_T SLEEP_H;

    SUBPOPULATION = AGE60 EQ 1 AND KDMOUT EQ 0;

    MISSING = ALL(-9999);
    IDVARIABLE = SEQN;
    STRATIFICATION = STRATUM;
    CLUSTER        = PSU;
    WEIGHT         = WT4YR;

ANALYSIS:
    TYPE        = COMPLEX;
    ESTIMATOR   = MLR;
    INFORMATION = OBSERVED;

MODEL:
    COG BY MEM_Z*   (l1)
           FLUEN_Z* (l2)
           SPEED_Z* (l3);
    COG@1;

    BA_ADV  ON IV (a1)
               AGE SEX BMI EDU WEAR_T SLEEP_H;
    SII_LOG ON IV (a2)
               AGE SEX BMI EDU WEAR_T SLEEP_H;
    COG ON IV       (cp)
           BA_ADV   (b1)
           SII_LOG  (b2)
           AGE SEX BMI EDU WEAR_T SLEEP_H;
    BA_ADV WITH SII_LOG;

MODEL INDIRECT:
    COG IND IV;
    COG IND BA_ADV IV;
    COG IND SII_LOG IV;

MODEL CONSTRAINT:
    NEW(ind_BA ind_SII total_ind total);
    l1 > 0;   ! fixes the orientation of the factor (COG@1 identifies it only up to sign)
    ind_BA    = a1 * b1;
    ind_SII   = a2 * b2;
    total_ind = ind_BA + ind_SII;
    total     = cp + total_ind;

OUTPUT:
    STDYX CINTERVAL;
'''

(MODELS / 'S1_depression_trimmed.inp').write_text(dep_inp)
(MODELS / 'S1_cognition_trimmed.inp').write_text(cog_inp)
print(f'Wrote {MODELS / "S1_depression_trimmed.inp"}')
print(f'Wrote {MODELS / "S1_cognition_trimmed.inp"}')
print()
print('Run both in Mplus, then compare the indirect effects with Table 4.')
print('Note: the mediation proportion is omitted from MODEL CONSTRAINT')
print('here, because the cognition total effect approaches zero and the ratio is')
print('therefore unstable.')